# Suite TST — The TypeScript Types dialect

TypeScript's types, typescript-estree's type nodes, the sublanguage that `as`, `satisfies`, `<T>x`, an annotation, a
type argument and a type parameter's constraint and default hold: the dialect `Expressions.TypeScript.Types` of the
TypeScript family, whose `Term` was TypeScript5's `Type` union, its branches unchanged, its kinds mbse-programs' 37 type
nodes, its holder, its binders refining `Let`, its editions, its printing, and its evaluation, which refuses each kind by
name (the dialects proposal, Retrofit's design; mbse-basis
[CAP-228: the TypeScript5 expression dialect](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-228.md)).

In [ ]:
import functools

from mbse.Expressions import Framework
from mbse.Expressions.IEEE754 import Expressions as IEEE754
from mbse.Expressions.ISOIEC11404 import Expressions as ISOIEC11404
from mbse.Expressions.Unicode import Expressions as Unicode
from mbse.Expressions.TypeScript import Patterns as TypeScriptPatterns, TypeScript5, Types as TypeScriptTypes
from mbse.Expressions.TypeScript.Patterns import Expressions as TSP
from mbse.Expressions.TypeScript.TypeScript5 import Evaluators as TV, Expressions as TS, Text as TSText
from mbse.Expressions.TypeScript.Types import Expressions as TST, Text as TSTText
from mbse.Expressions.Framework import Errors as FE, Standards as FS, Symbolics as SY, Terms as F
from mbse.Schemas.Framework import Builders, JSON, Validators, YAML
from support import raises

store = Builders.Store()  # the framework, the standard dialects TypeScript5's domains use, and TypeScript's family
for install in [Framework.install, ISOIEC11404.install, IEEE754.install, Unicode.install, TypeScript5.install,
                TypeScriptTypes.install, TypeScriptPatterns.install]:
    store.transaction(install)
store.transaction(TypeScriptTypes.define)  # the versions of TypeScript 5 the Types dialect covers
Term = store.registered("Expressions.Framework.Term")


READER = {d.union: d for d in (TS.DIALECT, TST.DIALECT, TSP.DIALECT)}  # the dialect whose union holds a term


def held(spec, dialect=TST.DIALECT):
    """The holder of the term `spec` writes, `dialect`'s own (the Types dialect's unless given), written in a
    transaction."""
    with store.transaction():
        return functools.reduce(getattr, dialect.path.split("."), store.Expressions).Expression(
            lambda e: e.root(dialect.spec(spec)))


def typed(spec):
    """A term of the Types dialect, held by its holder."""
    return F.term(held(spec))


def ts(spec):
    """A term of TypeScript5, held by its holder."""
    return F.term(held(spec, TS.DIALECT))


def reference(name):
    """A spec of a reference to the type `name`, in any union that has type references."""
    return lambda t: t.tstypereference(lambda r: r.typeName(lambda n: n.identifier(lambda i: i.name(name))))


T_, U_, K_ = reference("T"), reference("U"), reference("K")
number = lambda t: t.tsnumberkeyword(lambda k: k)  # noqa: E731
x = lambda e: e.identifier(lambda i: i.name("x"))  # noqa: E731: `x`, in any union that has identifiers

<a id="TST-01"></a>
## TST-01 · The Types dialect's union, its kinds' schemas in mbse-programs' shapes, its helpers, the positions that hold it, and its holder: validated, linked by `HasRoot`, round-tripped through JSON and YAML

In [ ]:
# Its Term: mbse-programs' 37 type nodes, its own, then the six terms of TypeScript5 a type holds, and invalid syntax, as
# TypeScript5's `Type` union had them; each own kind's schema is named by its path, in mbse-programs' case.
union = [b.name for b in store.registered(TST.DIALECT.union).branches]
assert union == [*TST.TAGS, *TST.TERMS, "error", "missing"] and len(TST.TAGS) == 37
assert TST.TERMS == ("identifier", "thisexpression", "literal", "unaryexpression", "templateliteral", "objectexpression")
assert TST.DIALECT.name() == "TypeScript Types" and TST.DIALECT.tag == "typescript_types"
assert TST.DIALECT.path == "TypeScript.Types" and TST.DIALECT.union == "Expressions.TypeScript.Types.Term"
kinds = {b.name: b.schema for b in store.registered(TST.DIALECT.union).branches}
assert [kinds[tag].name for tag in TST.TAGS[:3]] == [
    "Expressions.TypeScript.Types.TSAnyKeyword", "Expressions.TypeScript.Types.TSUnknownKeyword",
    "Expressions.TypeScript.Types.TSNumberKeyword"]
assert all(kinds[tag].name.startswith("Expressions.TypeScript.Types.TS") for tag in TST.TAGS)
assert all(kinds[tag].name.startswith("Expressions.TypeScript.TypeScript5.") for tag in TST.TERMS)  # TypeScript5's
assert kinds["identifier"] is {b.name: b.schema for b in store.registered(TS.DIALECT.union).branches}["identifier"]
assert {tag: list(kinds[tag].properties) for tag in ("tsqualifiedname", "tstypereference", "tsmappedtype", "tsfunctiontype",
                                                     "tsconditionaltype", "tsimporttype", "tstypepredicate")} == {
    "tsqualifiedname": ["left", "right"], "tstypereference": ["typeName", "typeArguments"],
    "tsmappedtype": ["name", "value", "body", "readonly", "key", "nameType", "optional"],  # Let's first
    "tsfunctiontype": ["name", "value", "body", "typeParameters", "params", "returnType"],
    "tsconditionaltype": ["checkType", "extendsType", "trueType", "falseType"],
    "tsimporttype": ["source", "options", "qualifier", "typeArguments"],
    "tstypepredicate": ["asserts", "parameterName", "typeAnnotation"]}
# Its helpers, only its kinds hold them: an object type's members, the union of its signatures and comments; TypeScript5
# keeps the helpers expressions hold too (an annotation, a type parameter, its declaration and instantiation).
assert TST.HELPERS == ("TSPropertySignature", "TSMethodSignature", "TSCallSignatureDeclaration",
                       "TSConstructSignatureDeclaration", "TSIndexSignature", "Comment")
TypeElement = store.registered("Expressions.TypeScript.Types.TypeElement")
assert [b.name for b in TypeElement.branches] == ["tspropertysignature", "tsmethodsignature", "tscallsignaturedeclaration",
                                                  "tsconstructsignaturedeclaration", "tsindexsignature", "comment"]
assert [b.schema.name for b in TypeElement.branches] == [f"Expressions.TypeScript.Types.{h}" for h in TST.HELPERS]
assert kinds["tstypeliteral"].properties["members"].schema.element is TypeElement
for name in ("TSTypeAnnotation", "TSTypeParameter", "TSTypeParameterDeclaration", "TSTypeParameterInstantiation"):
    assert store.registered(f"Expressions.TypeScript.TypeScript5.{name}") is not None
# The framework reads a type as a term: a kind's slots and containers holding its Term are its arguments; its helpers
# hold them too, where TypeScript5's annotation and instantiation hold a type.
Types = store.registered(TST.DIALECT.union)
assert F._parts(kinds["tsuniontype"]) == ((), ("types",), ()) and F._parts(kinds["tsarraytype"]) == (("elementType",), (), ())
assert F._parts(kinds["tsmappedtype"]) == (("value", "body", "nameType"), (), ("name", "readonly", "key", "optional"))
assert F._parts(kinds["tsfunctiontype"]) == (("value", "body"), ("params",), ("name", "typeParameters", "returnType"))
assert kinds["tsfunctiontype"].properties["params"].schema.element is store.registered(TSP.DIALECT.union)  # Patterns'
annotation = store.registered("Expressions.TypeScript.TypeScript5.TSTypeAnnotation")
assert annotation.properties["typeAnnotation"].schema is Types and F._parts(annotation) == (("typeAnnotation",), (), ())
assert store.registered("Expressions.TypeScript.TypeScript5.TSTypeParameterInstantiation").properties[
    "params"].schema.element is Types
# Where TypeScript5 holds a type: `as`, `satisfies` and `<T>x` hold its Term; an annotation and a type argument too
expressions = {b.name: b.schema for b in store.registered(TS.DIALECT.union).branches}
assert all(expressions[tag].properties["typeAnnotation"].schema is Types
           for tag in ("tsasexpression", "tssatisfiesexpression", "tstypeassertion"))
# Its holder, refining the framework's Expression, its root narrowed to its Term: validated, linked by HasRoot,
# round-tripped
Holder = store.registered("Expressions.TypeScript.Types.Expression")
assert Holder.ref and Holder.refines is store.registered("Expressions.Framework.Expression") and F.is_expression(Holder)
assert Holder.properties["root"].schema is Types and list(Holder.adjacencies) == ["abi"] and Holder.validate() == []
mapped = held(lambda t: t.tsmappedtype(lambda m: m.key(lambda i: i.name("K")).value(T_).body(K_)))
assert Validators.Validate(store)(Holder, mapped) == [] and TST.DIALECT.validate(mapped) == []
with store.transaction():
    linked = store.Schemas.OfExpression(lambda s: s.name("tst.mapped").expression(lambda a: a.insert(lambda e: e.expression(mapped))))
assert [F.same(entry.expression, mapped) for entry in Builders.OfExpression.Builder(linked, False, store).expression] == [True]
for encode, decode in [(JSON.ToJSON(store), JSON.FromJSON(store)), (YAML.ToYAML(store), YAML.FromYAML(store))]:
    assert F.same(decode(Holder, encode(Holder, mapped)).root, mapped.root)
    assert F.same(decode(Types, encode(Types, F.term(mapped))), F.term(mapped))
# Its install runs before or after its family's: each names the others' unions and helpers, created where not installed
alone = Builders.Store()
for install in [Framework.install, TypeScriptPatterns.install, TypeScriptTypes.install, TypeScript5.install]:
    alone.transaction(install)
assert [b.name for b in alone.registered(TST.DIALECT.union).branches] == union
assert list(alone.registered("Expressions.TypeScript.TypeScript5.TSTypeAnnotation").properties) == ["typeAnnotation"]

<a id="TST-02"></a>
## TST-02 · Each kind's checks, and an identifier in a type, a reference never reported unbound; forms and `make`; inference and evaluation, which refuse or know nothing of a type

In [ ]:
# Each kind's own checks, as mbse-programs has them, within a type held by its holder or by an expression
for spec, problems in [
    (lambda t: t.tsuniontype(lambda u: u.types([T_])), ["a tsuniontype needs two types"]),
    (lambda t: t.tsintersectiontype(lambda u: u.types([T_, U_])), []),
    (lambda t: t.tstemplateliteraltype(lambda u: u.quasis([lambda q: q.raw("a")]).types([T_])),
     ["a tstemplateliteraltype needs one more quasi than types"]),
    (lambda t: t.tsarraytype(lambda a: a), ["a tsarraytype needs an elementType"]),  # a slot it needs
    (lambda t: t.tsmappedtype(lambda m: m.key(lambda i: i.name("K"))), ["a tsmappedtype needs a value"]),
    (lambda t: t.tsmappedtype(lambda m: m.key(lambda i: i.name("K K")).value(T_)), ["an identifier's name cannot be 'K K'"]),
    (lambda t: t.tstypeoperator(lambda o: o.operator("keyof")), []),  # its type, optional
    (lambda t: t.tsimporttype(lambda i: i.source(lambda s: s.literal(lambda li: li.value(lambda v: v.as_scalar("m"))))), []),
    (lambda t: t.tsfunctiontype(lambda f: f.typeParameters(lambda d: d.params([]))), ["a tstypeparameterdeclaration needs a type parameter"]),
    (lambda t: t.tsfunctiontype(lambda f: f.typeParameters(lambda d: d.params([lambda p: p.name(lambda i: i.name("T"))]))), []),
    (lambda t: t.tsfunctiontype(lambda f: f.params([lambda p: p.identifier(lambda i: i.name("a b"))])),
     ["argument 0: an identifier's name cannot be 'a b'"]),  # a parameter, a Patterns term
    (lambda t: t.tstypeliteral(lambda q: q.members([lambda m: m.tspropertysignature(lambda s: s.typeAnnotation(
        lambda a: a.typeAnnotation(T_)))])), ["a tstypeliteral needs a members 0 key"]),  # a helper's slot, by its path
    (lambda t: t.tsfunctiontype(lambda f: f.returnType(lambda a: a)), ["a tsfunctiontype needs a returnType typeAnnotation"]),
    (lambda t: t.tsliteraltype(lambda q: q.literal(lambda li: li.literal(lambda v: v.value(lambda n: n.as_scalar("s")).quote("`")))),
     ["literal: a literal's quote cannot be '`'"]),  # a term of TypeScript5's, as it reads it
    (lambda t: t.tsconditionaltype(lambda c: c.checkType(T_).extendsType(U_).trueType(T_)), ["a tsconditionaltype needs a falseType"]),
    (lambda t: t.missing(lambda m: m.kind("type")), ["missing type"]),
    (lambda t: t.tsfunctiontype(lambda f: f.params([lambda p: p.identifier(lambda i: i.name("a").typeAnnotation(
        lambda a: a.typeAnnotation(lambda e: e.error(lambda r: r.text("1 1")))))])),
     ["argument 0: invalid syntax '1 1'"]),  # a type within a parameter's annotation, which its identifier checks
    (lambda t: t.tsfunctiontype(lambda f: f.params([lambda p: p.identifier(lambda i: i.name("a").typeAnnotation(lambda a: a))])), []),
]:
    assert TST.DIALECT.validate(typed(spec)) == problems, (TST.DIALECT.validate(typed(spec)), problems)
# An identifier in a type names a type, or a value whose type `typeof` reads, declared by what a term does not model: a
# reference, never reported unbound, nor free; its name is checked as an identifier's
assert TST.DIALECT.kinds()["identifier"].role == F.REFERENCE and not TST.DIALECT.kinds()["identifier"].lexical
assert TST.DIALECT.kind_named("identifier") is not TS.DIALECT.kinds()["identifier"] and TS.DIALECT.kinds()["identifier"].lexical
query = typed(lambda t: t.tstypequery(lambda q: q.exprName(lambda e: e.identifier(lambda i: i.name("value")))))
assert TST.DIALECT.validate(query) == [] and SY.free(query, TST.DIALECT) == set() and TS.DIALECT.validate(query) == []
assert TST.DIALECT.validate(typed(lambda t: t.tstypereference(lambda r: r.typeName(lambda n: n.identifier(lambda i: i.name("1")))))) == [
    "typeName: an identifier's name cannot be '1'"]
cast = ts(lambda u: u.tsasexpression(lambda r: r.expression(x).typeAnnotation(T_)))  # `x as T`: x a value's name, T a type's
assert TS.DIALECT.validate(cast) == ["expression: identifier 'x' is not bound"] and SY.free(cast, TS.DIALECT) == {"x"}
# Its kinds read and written by their forms: a type's arguments are its slots, its containers' terms, its helpers'
every = typed(lambda t: t.tsuniontype(lambda u: u.types([*[lambda k, keyword=keyword: getattr(k, keyword)(lambda h: h)
                                                            for keyword in TST.KEYWORDS], T_,
    lambda k: k.tstupletype(lambda p: p.elementTypes([lambda e: e.tsnamedtuplemember(lambda m: m.label(lambda i: i.name("a")).elementType(T_))])),
    lambda k: k.tsconditionaltype(lambda c: c.checkType(T_).extendsType(lambda e: e.tsinfertype(lambda i: i.typeParameter(
        lambda p: p.name(lambda n: n.name("V"))))).trueType(T_).falseType(U_))])))
for node in F.walk(every):
    reader = READER[F.dialect_of(node)]  # made by the dialect whose union holds it, in its holder
    assert F.same(F.term(held(reader.make(F.form(node)), reader)), node)
assert F.form(typed(lambda t: t.tsarraytype(lambda a: a.elementType(T_)))).arguments[0] is not None
form = F.form(typed(T_))
assert form.kind == "tstypereference" and form.attributes == {} and len(form.arguments) == 1  # its name, a term
assert F.form(form.arguments[0]) == F.Form("identifier", {"name": "T"})
assert F.form(typed(lambda t: t.tsfunctiontype(lambda f: f.typeParameters(lambda d: d.params([lambda p: p.name(lambda i: i.name("T"))]))))) == F.Form(
    "tsfunctiontype", {"typeParameters": F.Part("tstypeparameterdeclaration", {"params": (F.Part("tstypeparameter", {
        "name": F.Part("identifier", {"name": "T"}, 0)}, 2),)}, 2)}, (None, None, None, None))  # Let's empty, then a type parameter's
with raises(ValueError, match="TypeScript Types has no kind 'call'"):
    TST.DIALECT.make(F.Form("call"))
# Inference knows nothing of a type; evaluation refuses each kind by its dialect's name, as TypeScript5's evaluator does
assert TST.DIALECT.infer(typed(T_)).name() == "any"
for spec, message in [(T_, "TypeScript Types tstypereference is not evaluated"),
                      (lambda t: t.tsanykeyword(lambda h: h), "TypeScript Types tsanykeyword is not evaluated"),
                      (lambda t: t.identifier(lambda i: i.name("T")), "TypeScript Types identifier is not evaluated")]:
    with raises(NotImplementedError, match=message):
        TV.OfAny(typed(spec))
with raises(ValueError, match="TypeScript Types missing is not evaluated"):
    TV.OfAny(typed(lambda t: t.missing(lambda m: m)))
assert TV.OfAny(ts(lambda u: u.literal(lambda r: r.value(lambda v: v.as_scalar(1.0))))) == 1.0  # TypeScript5's, as before

<a id="TST-03"></a>
## TST-03 · Its binders refine `Let`: a mapped type's constraint and type, Let's `value` and `body`, its key bound in its name type and its type; a function type's and a constructor type's type parameters and parameters within them; a conditional type's `infer` declarations in its true type, by its scopes; a signature's parameters within it

In [ ]:
LET = store.registered("Expressions.Framework.Let")
assert {tag for tag in TST.TAGS if F.is_binder(kinds[tag])} == {"tsmappedtype", "tsfunctiontype", "tsconstructortype"}
assert all(kinds[tag].refines is LET for tag in ("tsmappedtype", "tsfunctiontype", "tsconstructortype"))
assert kinds["tsmappedtype"].properties["value"].schema is kinds["tsmappedtype"].properties["body"].schema is Types  # its own
assert kinds["tsfunctiontype"].properties["value"].schema is Term  # Let's, unset
assert not F.is_binder(kinds["tsconditionaltype"]) and not F.is_binder(TypeElement.branches[1].schema)
assert all(TST.DIALECT.kinds()[tag].scopes is not None for tag in TST.TAGS)  # each binds by its scopes, none as a let
# What each argument has bound: a mapped type's key in its name type and its type, not in its constraint
mapped = typed(lambda t: t.tsmappedtype(lambda m: m.key(lambda i: i.name("K")).value(K_).body(K_).nameType(K_)))
assert TST.DIALECT.kinds()["tsmappedtype"].scopes(F.kind_of(mapped)[1]) == [frozenset(), {"K"}, {"K"}]
assert F.form(mapped).attributes == {"key": F.Part("identifier", {"name": "K"}, 0)} and len(F.form(mapped).arguments) == 3
assert TST.DIALECT.validate(mapped) == [] and SY.free(mapped, TST.DIALECT) == set()
function = typed(lambda t: t.tsfunctiontype(lambda f: f.typeParameters(lambda d: d.params([lambda p: p.name(lambda i: i.name("T"))])).params([
    lambda p: p.identifier(lambda i: i.name("a").typeAnnotation(lambda b: b.typeAnnotation(T_))),
    lambda p: p.restelement(lambda r: r.argument(lambda i: i.identifier(lambda j: j.name("rest"))))]).returnType(
    lambda r: r.typeAnnotation(lambda p: p.tstypepredicate(lambda q: q.parameterName(lambda n: n.identifier(lambda i: i.name("a"))))))))
scopes = TST.DIALECT.kinds()["tsfunctiontype"].scopes(F.kind_of(function)[1])
assert len(scopes) == len(F.form(function).arguments) == 7 and set(scopes) == {frozenset({"T", "a", "rest"})}
assert TST.DIALECT.validate(function) == [] and SY.free(function, TST.DIALECT) == set()  # its parameters bound within it
constructor = typed(lambda t: t.tsconstructortype(lambda f: f.params([x]).returnType(lambda r: r.typeAnnotation(T_))))
assert TST.DIALECT.kinds()["tsconstructortype"].scopes(F.kind_of(constructor)[1]) == [{"x"}] * 4
# A conditional type binds its `infer` declarations in its true type, but those of a conditional type within its extends
conditional = typed(lambda t: t.tsconditionaltype(lambda c: c.checkType(T_).extendsType(lambda e: e.tsarraytype(lambda a: a.elementType(
    lambda i: i.tsinfertype(lambda f: f.typeParameter(lambda p: p.name(lambda n: n.name("V")))))))
    .trueType(reference("V")).falseType(lambda e: e.tsconditionaltype(lambda d: d.checkType(T_).extendsType(
        lambda i: i.tsinfertype(lambda f: f.typeParameter(lambda p: p.name(lambda n: n.name("W"))))).trueType(T_).falseType(U_)))))
assert TST.DIALECT.kinds()["tsconditionaltype"].scopes(F.kind_of(conditional)[1]) == [frozenset(), frozenset(), {"V"}, frozenset()]
nested = typed(lambda t: t.tsconditionaltype(lambda c: c.checkType(T_).extendsType(lambda e: e.tsconditionaltype(lambda d: d.checkType(T_)
    .extendsType(lambda i: i.tsinfertype(lambda f: f.typeParameter(lambda p: p.name(lambda n: n.name("W"))))).trueType(T_).falseType(U_)))
    .trueType(T_).falseType(U_)))
assert TST.DIALECT.kinds()["tsconditionaltype"].scopes(F.kind_of(nested)[1])[2] == frozenset()  # the inner one's own
for extends in [lambda i: i.tsinfertype(lambda f: f.typeParameter(lambda p: p)), lambda i: i.tsinfertype(lambda f: f), None]:
    written = lambda c, extends=extends: c.checkType(T_).extendsType(extends) if extends is not None else c.checkType(T_)  # noqa: E731
    anonymous = typed(lambda t: t.tsconditionaltype(lambda c: written(c).trueType(T_).falseType(U_)))
    assert TST.DIALECT.kinds()["tsconditionaltype"].scopes(F.kind_of(anonymous)[1])[-2] == frozenset()  # an infer of no
    # name, or none, binds none
# A mapped type with no key, or a key of no name, binds none; a type parameter of no name declares none
for key in [lambda m: m, lambda m: m.key(lambda i: i)]:
    unnamed = typed(lambda t: t.tsmappedtype(lambda m: key(m).value(K_).body(K_)))
    assert TST.DIALECT.kinds()["tsmappedtype"].scopes(F.kind_of(unnamed)[1]) == [frozenset()] * 3
generic = typed(lambda t: t.tsfunctiontype(lambda f: f.typeParameters(lambda d: d.params([lambda p: p, lambda p: p.name(lambda i: i)]))))
assert set(TST.DIALECT.kinds()["tsfunctiontype"].scopes(F.kind_of(generic)[1])) == {frozenset()}
# A signature binds its parameters within it, through the type literal that holds it; a key, not computed, is a name
literal = typed(lambda t: t.tstypeliteral(lambda q: q.members([
    lambda m: m.tsmethodsignature(lambda s: s.key(lambda k: k.identifier(lambda i: i.name("m"))).params([x]).returnType(
        lambda r: r.typeAnnotation(lambda p: p.tstypequery(lambda e: e.exprName(lambda n: n.identifier(lambda i: i.name("x"))))))),
    lambda m: m.tsindexsignature(lambda s: s.parameters([lambda p: p.identifier(lambda i: i.name("k"))])),
    lambda m: m.tspropertysignature(lambda s: s.computed(True).key(x)),
    lambda m: m.tscallsignaturedeclaration(lambda s: s.params([lambda p: p.identifier(lambda i: i.name("y"))]))])))
assert TST.DIALECT.kinds()["tstypeliteral"].scopes(F.kind_of(literal)[1]) == [{"m", "x"}, {"x"}, {"x"}, {"k"}, frozenset(), {"y"}]
assert TST.DIALECT.validate(literal) == ["members 2 key: identifier 'x' is not bound"]  # a computed key, a value's name
assert SY.free(literal, TST.DIALECT) == {"x"}

<a id="TST-04"></a>
## TST-04 · Its editions: the versions of TypeScript 5, 5.0 to 5.9, by their published names, each with every one of its own kinds

In [ ]:
tst = store.Instances.Expressions.TypeScript.Types
VERSIONS = tuple(f"TypeScript 5.{minor}" for minor in range(10))
assert tuple(FS.standards(tst)) == VERSIONS and TST.SINCE == {}
assert FS.properties(FS.find(tst, "TypeScript 5.4")) == {"bodies": ("com.microsoft",), "revision": ("version", "5.4")}
assert [s.name for s in FS.kinds(FS.find(tst, "TypeScript 5.0"))] == [kinds[tag].name for tag in TST.TAGS]  # its own
store.transaction(TypeScriptTypes.define)  # again: it writes nothing
assert tuple(FS.standards(tst)) == VERSIONS

<a id="TST-05"></a>
## TST-05 · Printed as TypeScript writes types, by precedence, from its own holder and within TypeScript5's, by the family's printer

In [ ]:
assert TSTText.ToText is TSText.ToText and set(TSTText.TEXTS) == set(TST.TAGS) | {
    "tspropertysignature", "tsmethodsignature", "tscallsignaturedeclaration", "tsconstructsignaturedeclaration", "tsindexsignature"}
assert TSTText.PRECEDENCE["tsuniontype"] == TSTText.UNION and TSTText.FUNCTION < TSTText.CONDITIONAL < TSTText.PRIMARY
for spec, text in [
    (lambda t: t.tsuniontype(lambda u: u.types([T_, lambda e: e.tsfunctiontype(lambda f: f.returnType(lambda r: r.typeAnnotation(U_)))])),
     "T | (() => U)"),  # a function type as an operand, parenthesized
    (lambda t: t.tsmappedtype(lambda m: m.readonly("-readonly").key(lambda i: i.name("K")).value(lambda e: e.tstypeoperator(
        lambda o: o.operator("keyof").typeAnnotation(T_))).optional("-?").body(lambda e: e.tsindexedaccesstype(
            lambda a: a.objectType(T_).indexType(K_)))), "{ -readonly [K in keyof T]-?: T[K] }"),
    (lambda t: t.tsconstructortype(lambda f: f.typeParameters(lambda d: d.params([lambda p: p.name(lambda i: i.name("T"))])).params([
        lambda p: p.identifier(lambda i: i.name("a"))]).returnType(lambda r: r.typeAnnotation(T_))), "new <T>(a) => T"),
    (lambda t: t.tstypeliteral(lambda q: q.members([
        lambda m: m.tsmethodsignature(lambda s: s.key(lambda k: k.identifier(lambda i: i.name("m"))).params([x]).returnType(
            lambda r: r.typeAnnotation(T_))),
        lambda m: m.tsindexsignature(lambda s: s.readonly(True).parameters([lambda p: p.identifier(lambda i: i.name("k").typeAnnotation(
            lambda a: a.typeAnnotation(lambda k: k.tsstringkeyword(lambda h: h))))]).typeAnnotation(lambda a: a.typeAnnotation(T_)))])),
     "{ m(x): T; readonly [k: string]: T; }"),
    (lambda t: t.tsconditionaltype(lambda c: c.checkType(T_).extendsType(lambda e: e.tsarraytype(lambda a: a.elementType(
        lambda i: i.tsinfertype(lambda f: f.typeParameter(lambda p: p.name(lambda n: n.name("V"))))))).trueType(reference("V")).falseType(
        lambda e: e.tsneverkeyword(lambda h: h))), "T extends (infer V)[] ? V : never"),
    (lambda t: t.tsimporttype(lambda i: i.source(lambda s: s.literal(lambda li: li.value(lambda v: v.as_scalar("m")))).qualifier(
        lambda q: q.tsqualifiedname(lambda n: n.left(lambda l: l.identifier(lambda j: j.name("A"))).right(lambda j: j.name("B"))))),
     'import("m").A.B'),
    (lambda t: t.error(lambda e: e.text("T T")), "T T"),  # invalid syntax, as written
]:
    assert TSText.ToText(typed(spec)) == text, (TSText.ToText(typed(spec)), text)
    assert TSText.ToText(held(spec)) == text  # from its holder
with raises(ValueError, match="TypeScript Types error has no text to print"):  # refused by its dialect's name
    TSText.ToText(typed(lambda t: t.error(lambda e: e)))
# Within TypeScript5's terms, an annotation, a type argument, and a type parameter's constraint and default
assert TSText.ToText(ts(lambda u: u.arrowfunctionexpression(lambda a: a.typeParameters(lambda d: d.params([
    lambda p: p.name(lambda i: i.name("T")).constraint(lambda e: e.tsuniontype(lambda n: n.types([number, U_]))).default(number)]))
    .params([lambda p: p.identifier(lambda i: i.name("v").typeAnnotation(lambda b: b.typeAnnotation(T_)))]).returnType(
        lambda r: r.typeAnnotation(lambda e: e.tsfunctiontype(lambda f: f.returnType(lambda s: s.typeAnnotation(T_)))))
    .body(lambda b: b.callexpression(lambda c: c.callee(x).typeArguments(lambda g: g.params([T_]))))))) == (
    "<T extends number | U = number>(v: T): () => T => x<T>()")